# 01 — Heaps and Priority Queues

## Why This Matters for DSA
Heaps open Phase 03 because they're the simplest hierarchical structure — a binary tree with exactly one ordering rule (parent vs children), stored without any pointers at all. That simplicity is deceptive: heaps are the backbone of an entire family of "keep finding the best/worst thing efficiently, as data keeps changing" problems — top-k, running statistics on a stream, merging many sorted sources, and Dijkstra's algorithm later in the graphs phase. `03_STL_Deep_Dive` introduced the mechanics; this notebook is about the PATTERNS built on top of them.

## Prerequisites
- `03_STL_Deep_Dive` (Phase 00), Section 5.6 — heap array-index math, `priority_queue` basics
- `04_Sorting_Algorithms` (Phase 02), Section 4 — heap sort's build-heap step, reused directly here
- `02_Sliding_Window_and_Prefix_Sum` (Phase 02) — the two-pointer merge pattern, generalized to k-way merge in this notebook

## Learning Objectives
By the end of this notebook, you will be able to:
- Explain why building a heap from an existing array is O(n), not O(n log n), and why that distinction actually matters
- Write custom comparators for `priority_queue` on user-defined types, both via `operator<` and via a separate comparator
- Apply the top-k pattern (bounded-size heap) to avoid a full sort when only a few extreme elements matter
- Apply the two-heap technique to maintain a running median in O(log n) per insertion
- Apply k-way merge to combine multiple sorted sources in O(N log k) instead of O(N log N)
- Judge when a heap is the right structure versus when sorting, a hash-based approach, or something else fits better

## Checklist Before Moving On
- [ ] I can state why build-heap is O(n) despite each individual heapify call being O(log n) in the worst case
- [ ] I can write a `priority_queue<CustomStruct>` with a correct `operator<` without second-guessing the direction
- [ ] I can implement the top-k pattern and explain why it uses a MIN-heap to find the Kth LARGEST element
- [ ] I can implement the two-heap median technique and explain the size-balancing invariant
- [ ] I can implement k-way merge and state its complexity relative to concatenate-then-sort
- [ ] I can identify when a heap is the wrong tool — for example, when you need to search for an arbitrary element, not just repeatedly access an extreme one

## Table of Contents
1. Heap Fundamentals Recap — The Complete Binary Tree Property
2. Build-Heap in O(n) — Why It's Not O(n log n)
3. Custom Comparators — Ordering Complex Types in a Priority Queue
4. The Top-K Pattern — Finding the Kth Largest Without a Full Sort
5. The Two-Heap Technique — Running Median
6. K-Way Merge — Merging Multiple Sorted Lists With a Heap
7. When a Heap Is (and Isn't) the Right Tool
8. Phase Checkpoint, Cheat Sheet, and Answer Key
9. LeetCode Practice Problems


## 1. Heap Fundamentals Recap — The Complete Binary Tree Property

### The Why
Everything in this notebook builds on the array-based heap mechanics from `03_STL_Deep_Dive`. This is a deliberately brief recap — read that section first if any of this feels unfamiliar — establishing the vocabulary this notebook uses throughout.

### Core Mechanism
- A heap is a **complete** binary tree: every level is completely filled except possibly the last, which fills left to right with no gaps. This completeness is exactly what allows array-based storage with no pointers: node `i`'s parent is at `(i-1)/2`, its children are at `2i+1` and `2i+2`.
- **Max-heap:** every parent ≥ its children — the maximum element is always at the root (index 0). **Min-heap:** every parent ≤ its children — the minimum is always at the root.
- `std::priority_queue<T>` is a max-heap by default; pass `greater<T>` as the third template argument for a min-heap: `priority_queue<T, vector<T>, greater<T>>`.
- Push and pop are both O(log n) — a new element bubbles up to its correct position, or the root's replacement (the last element, moved to the root) sinks down to its correct position.

### Common Pitfalls
- If any of this recap feels genuinely new rather than familiar, pause and work through `03_STL_Deep_Dive` Section 5.6 first — this notebook assumes that mechanical foundation and focuses entirely on patterns built on top of it.


## 2. Build-Heap in O(n) — Why It's Not O(n log n)

### The Why
This is a genuinely counter-intuitive result worth proving rather than just stating: building a heap from `n` existing elements is O(n), even though each individual `heapify` call can cost up to O(log n) in the worst case. The naive assumption — "n calls, each up to O(log n), so O(n log n) total" — is wrong, and understanding WHY reveals something real about how work is distributed across a tree.

### Core Mechanism
- **Bottom-up build** (the standard approach, same as `04_Sorting_Algorithms`' heap sort build step): call `heapify` on every non-leaf node, starting from the last non-leaf node and working backward to the root.
- **The key insight:** a node at height `h` (measuring from the bottom — leaves are height 0) can sink AT MOST `h` levels during its heapify call. Most nodes in a complete binary tree are near the BOTTOM — roughly half of all nodes are leaves (height 0, doing zero work), a quarter are at height 1 (doing at most 1 unit of work), and so on. Only a tiny fraction of nodes — a single one, actually — sits at the maximum height near the root, where the expensive O(log n) sink could even happen.
- Summing `(number of nodes at height h) × h` across all heights produces a series that converges to O(n), NOT O(n log n) — this is a standard but genuinely elegant result (the full derivation involves a converging geometric-like series; the takeaway to internalize is the CONCLUSION and the intuition for why, not necessarily memorizing the summation itself).
- The empirical measurement below confirms this directly: the ratio of comparisons to `n` stays roughly CONSTANT as `n` grows across two orders of magnitude — exactly the signature of O(n), using the same benchmarking instinct as `01_Complexity_Analysis`'s empirical verification section, just applied to a comparison count instead of wall-clock time.

### Common Pitfalls
- Assuming "n calls to a function that's O(log n) each" always multiplies out to O(n log n) — this is true only when every call actually costs close to its worst case. Build-heap is a clean counterexample: the vast majority of calls are cheap (near the leaves), and only a vanishing few are expensive (near the root).


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// BUILD-HEAP IN O(n), NOT O(n log n): inserting n elements one at a time (each insert
// being O(log n) via heapifyUp) gives O(n log n) total. But building a heap from an
// EXISTING array all at once -- via bottom-up heapify, exactly like heap sort's build
// step from 04_Sorting_Algorithms -- is actually O(n), not O(n log n). The proof is
// about WHERE the work happens, not just how many nodes get heapified.

long long comparisonsBottomUp = 0;

void heapifyDown(vector<int>& v, int n, int i) {
    int largest = i;
    int left = 2 * i + 1, right = 2 * i + 2;

    if (left < n) { comparisonsBottomUp++; if (v[left] > v[largest]) largest = left; }
    if (right < n) { comparisonsBottomUp++; if (v[right] > v[largest]) largest = right; }

    if (largest != i) {
        swap(v[i], v[largest]);
        heapifyDown(v, n, largest);
    }
}

void buildHeapBottomUp(vector<int>& v) {
    int n = (int)v.size();
    for (int i = n / 2 - 1; i >= 0; i--) {
        heapifyDown(v, n, i);
    }
}

int main() {
    for (int n : {1000, 10000, 100000}) {
        vector<int> v(n);
        for (int i = 0; i < n; i++) v[i] = n - i;   // reverse order -- a reasonable stress case

        comparisonsBottomUp = 0;
        buildHeapBottomUp(v);
        cout << "n=" << n << " bottom-up comparisons=" << comparisonsBottomUp
             << " (n=" << n << ", ratio=" << (double)comparisonsBottomUp / n << ")\n";
    }

    // WHY THIS IS O(n): most nodes in a complete binary tree are near the BOTTOM (leaves
    // and near-leaves), and a node at height h can sink AT MOST h levels during its
    // heapify call. Summing (number of nodes at height h) * h across all heights gives
    // a series that converges to O(n), NOT O(n log n) -- because the vast majority of
    // nodes are at small heights (close to 0), and only a few nodes are near the root
    // (height ~log n) where the expensive O(log n) sinks could even happen.
    // The ratio printed above stays roughly CONSTANT (close to 2) as n grows -- that's
    // the empirical signature of O(n), the same kind of check used in
    // 01_Complexity_Analysis's benchmarking section, just applied to a comparison count
    // instead of wall-clock time.

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Custom Comparators — Ordering Complex Types in a Priority Queue

### The Why
Real problems rarely put plain integers in a heap — they put tasks, points, intervals, states. Getting the comparator direction right for a `priority_queue` of a custom type trips people up constantly, because `priority_queue` is a MAX-heap by default, which means `operator<` sometimes needs to be defined "backwards" relative to plain intuition.

### Core Mechanism
- **Via `operator<` on the struct itself:** simplest when there's ONE natural priority ordering for the type. Remember `priority_queue` calls `.top()` on whatever `operator<` says is "biggest" — so if you want SMALLER priority numbers to come out first (more urgent = processed first), `operator<` must be defined so that a bigger priority NUMBER counts as "less than" — i.e., `return priority > other.priority;`, which looks backwards until you remember it's answering "is this LESS urgent," not "is this priority number smaller."
- **Via a separate comparator (lambda + `decltype`):** necessary when a type needs MULTIPLE different orderings in different contexts — `operator<` can only encode one ordering per type, but a lambda-based comparator lets you build as many differently-ordered `priority_queue`s over the same type as you need.
- Both approaches follow the exact same underlying rule: the comparator answers "does the FIRST argument have LOWER priority than the second" — get comfortable re-deriving the direction from that question rather than memorizing "it's backwards" as a rule of thumb without the reasoning behind it.

### Common Pitfalls
- Writing `operator<` the "intuitive" direction (`priority < other.priority`) when you actually want smaller numbers to be more urgent — this silently gives you a queue that processes the LEAST urgent tasks first, since `priority_queue` always surfaces whatever `operator<` considers biggest.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <queue>
#include <string>
#include <vector>
using namespace std;

// custom comparator for a priority_queue of a user-defined struct: two ways to do it --
// (1) overload operator< on the struct itself, or (2) supply a separate comparator.
// Overloading operator< is simplest when there's ONE natural priority ordering for the type.

struct Task {
    string name;
    int priority;   // lower number = more urgent

    // NOTE: priority_queue is a MAX-heap by default -- it calls .top() on whatever
    // operator< says is "biggest." Since we want LOWER priority numbers to come first
    // (more urgent = higher priority in the queue sense), operator< must be defined
    // BACKWARDS relative to normal intuition: a Task is "less than" another if its
    // priority number is actually BIGGER -- so the smallest-number task ends up "biggest"
    // by this operator, and is therefore what .top() returns.
    bool operator<(const Task& other) const {
        return priority > other.priority;   // reversed on purpose -- see note above
    }
};

int main() {
    priority_queue<Task> tasks;
    tasks.push({"Deploy hotfix", 1});
    tasks.push({"Write docs", 5});
    tasks.push({"Review PR", 2});
    tasks.push({"Fix critical bug", 0});

    cout << "processing by urgency:\n";
    while (!tasks.empty()) {
        cout << "  " << tasks.top().name << " (priority " << tasks.top().priority << ")\n";
        tasks.pop();
    }
    // expected order: Fix critical bug(0), Deploy hotfix(1), Review PR(2), Write docs(5)

    // ALTERNATIVE: a separate comparator lambda instead of operator<, useful when a type
    // needs MULTIPLE different orderings in different contexts (operator< can only encode one)
    auto byNameLength = [](const Task& a, const Task& b) {
        return a.name.size() > b.name.size();   // shorter names = higher priority
    };
    priority_queue<Task, vector<Task>, decltype(byNameLength)> byLength(byNameLength);
    byLength.push({"Deploy hotfix", 1});
    byLength.push({"Write docs", 5});
    byLength.push({"Review PR", 2});

    cout << "processing by shortest name first:\n";
    while (!byLength.empty()) {
        cout << "  " << byLength.top().name << "\n";
        byLength.pop();
    }

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. The Top-K Pattern — Finding the Kth Largest Without a Full Sort

### The Why
This is the highest-frequency practical heap pattern: whenever a problem only cares about the k largest (or smallest) elements, sorting the ENTIRE array is wasted work — a heap bounded to size k gets the answer for meaningfully less cost when k is small relative to n.

### Core Mechanism
- Maintain a heap capped at size `k`. For "find the Kth largest," use a MIN-heap (this is the detail that trips people up — see below).
- For each element: push it, then if the heap now exceeds size `k`, pop the minimum — evicting whatever is currently the WEAKEST member of the top-k candidates, since it can no longer possibly be in the true top-k once k stronger candidates exist.
- After processing everything, the heap's minimum (its `.top()`, since it's a min-heap) IS the Kth largest element — it's the smallest element among the k largest, which is exactly the definition of "Kth largest."
- **Why a MIN-heap for finding the Kth LARGEST (the counter-intuitive part):** the heap holds the current top-k CANDIDATES, and the operation you need to do repeatedly and fast is "find and evict the weakest candidate" — that's the MINIMUM of the top-k set, and a min-heap gives you O(1) access to its minimum via `.top()`.
- **Complexity:** O(n log k) — each of `n` elements does at most one O(log k) push and possibly one O(log k) pop, since the heap never exceeds size k. Compare to sorting everything first at O(n log n) — when k is small relative to n, O(n log k) is a meaningful improvement.

### Common Pitfalls
- Using a MAX-heap instead of a min-heap for this pattern — a max-heap of size k gives you fast access to the LARGEST of the top-k candidates, which is not what you need to decide who to evict when a new, potentially-better candidate arrives.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>
using namespace std;

// TOP-K PATTERN: find the Kth largest element (or the K largest elements) WITHOUT
// sorting the entire array. Maintain a MIN-heap of size k -- if a new element beats
// the heap's minimum (the current "weakest" member of the top-k so far), it replaces it.
int findKthLargest(vector<int>& nums, int k) {
    priority_queue<int, vector<int>, greater<int>> minHeap;   // min-heap, capped at size k

    for (int num : nums) {
        minHeap.push(num);
        if ((int)minHeap.size() > k) {
            minHeap.pop();   // evict the current smallest -- it can't be in the top k anymore
        }
    }
    return minHeap.top();   // after processing everything, the heap's minimum IS the kth largest
}

int main() {
    vector<int> nums{3, 2, 1, 5, 6, 4};
    cout << "3rd largest: " << findKthLargest(nums, 3) << "\n";   // expected 4

    vector<int> nums2{3, 2, 3, 1, 2, 4, 5, 5, 6};
    cout << "4th largest: " << findKthLargest(nums2, 4) << "\n";  // expected 4

    // COMPLEXITY: O(n log k) -- each of n elements does at most one O(log k) push and
    // possibly one O(log k) pop, since the heap never grows past size k. Compare to
    // sorting the whole array first (O(n log n)) -- when k is small relative to n, this
    // is a meaningful win: O(n log k) vs O(n log n), especially for large n and small k.
    // WHY A MIN-HEAP, not a max-heap, for "find the Kth LARGEST": counterintuitive at
    // first, but the heap holds the CURRENT top-k candidates, and you need FAST ACCESS
    // to the WEAKEST of those candidates (to know what to evict) -- that's the minimum
    // of the top-k set, which is exactly what a min-heap's .top() gives you in O(1)

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. The Two-Heap Technique — Running Median

### The Why
This is the pattern for maintaining a running STATISTIC that depends on "the middle" of a growing dataset — the median — without re-sorting everything on every new arrival. Two heaps working together give O(log n) per insertion, versus O(n log n) if you re-sorted the accumulated data every time.

### Core Mechanism
- Split all data seen so far into two halves using two heaps: a MAX-heap holding the SMALLER half of the values, a MIN-heap holding the LARGER half. The median lives right at the boundary between these two halves — accessible from their tops in O(1).
- **Insertion procedure:** always push the new value into `lowerHalf` (the max-heap) first, then immediately move `lowerHalf`'s current maximum into `upperHalf` — this two-step "insert, then shuffle the max across" keeps the logic uniform regardless of which half the new value actually belongs in, rather than needing a comparison to decide upfront.
- **The size-balancing invariant:** after every insertion, `lowerHalf`'s size must be either EQUAL to or exactly ONE MORE than `upperHalf`'s size — never behind, never more than one ahead. Maintaining this invariant is what makes the median formula simple: if sizes are unequal, the median is `lowerHalf`'s top directly (it holds the single middle element); if equal, the median is the average of both tops.
- **Complexity:** O(log n) per insertion (two heap operations, each O(log heap size)), O(1) to query the median at any point — versus O(n log n) PER insertion if you naively re-sorted everything seen so far.

### Common Pitfalls
- Forgetting the rebalancing step after the initial push/pop — without it, one heap can grow arbitrarily larger than the other, breaking the O(1) median-lookup formula, which depends entirely on the size invariant holding.
- Getting the median formula's odd/even case backwards — double check which heap is expected to hold the "extra" middle element in an odd-sized dataset (in this implementation, `lowerHalf`).


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <queue>
#include <vector>
using namespace std;

// TWO-HEAP TECHNIQUE: maintain the running median of a stream of numbers, with O(log n)
// per insertion. The trick: split the data into two halves using two heaps -- a MAX-heap
// holding the SMALLER half, a MIN-heap holding the LARGER half. The median is then
// always accessible from the tops of these two heaps in O(1), no full sort ever needed.
class MedianFinder {
    priority_queue<int> lowerHalf;                                   // max-heap: smaller half
    priority_queue<int, vector<int>, greater<int>> upperHalf;        // min-heap: larger half

public:
    void addNum(int num) {
        // always insert into lowerHalf first, then rebalance -- this two-step keeps the
        // logic simple and correct regardless of which half `num` actually belongs in
        lowerHalf.push(num);
        upperHalf.push(lowerHalf.top());
        lowerHalf.pop();

        // maintain the size invariant: lowerHalf has either the same count as upperHalf,
        // or exactly one more (never falls behind, never gets more than one ahead)
        if (upperHalf.size() > lowerHalf.size()) {
            lowerHalf.push(upperHalf.top());
            upperHalf.pop();
        }
    }

    double findMedian() {
        if (lowerHalf.size() > upperHalf.size()) {
            return lowerHalf.top();   // odd total count -- lowerHalf has the extra middle element
        }
        // even total count -- median is the average of both halves' tops
        return (lowerHalf.top() + upperHalf.top()) / 2.0;
    }
};

int main() {
    MedianFinder mf;
    vector<int> stream{5, 15, 1, 3, 8, 7, 9, 10, 20};

    for (int num : stream) {
        mf.addNum(num);
        cout << "after adding " << num << ": median=" << mf.findMedian() << "\n";
    }
    // sanity check against sorting the prefix each time: [5]->5, [5,15]->10, [1,5,15]->5,
    // [1,3,5,15]->4, [1,3,5,8,15]->5, [1,3,5,7,8,15]->6, [1,3,5,7,8,9,15]->7,
    // [1,3,5,7,8,9,10,15]->7.5, [1,3,5,7,8,9,10,15,20]->8

    // COMPLEXITY: O(log n) per insertion (two heap pushes/pops, each O(log heap size)),
    // O(1) to query the median at any point -- versus re-sorting the entire seen-so-far
    // list on every insertion, which would be O(n log n) PER insertion. This is the
    // pattern to reach for whenever a problem needs a running statistic that depends on
    // "the middle" of a growing dataset, not just a min or max

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. K-Way Merge — Merging Multiple Sorted Lists With a Heap

### The Why
`02_Sliding_Window_and_Prefix_Sum` covered merging exactly TWO sorted arrays with two pointers. This is the natural generalization to an arbitrary number of sorted sources at once — and a heap is what makes "which of the k current front elements is smallest" an O(log k) question instead of an O(k) one.

### Core Mechanism
- Seed a min-heap with the FIRST element of every list, along with enough bookkeeping (which list it came from, and its position within that list) to find that list's next element once this one is consumed.
- Repeatedly pop the heap's minimum (guaranteed to be the smallest "current front" across all k lists), append it to the result, and — if the list it came from has more elements — push that list's next element into the heap.
- **Why the heap beats a direct comparison:** without it, finding the smallest of k "current front" elements at each step requires comparing all k of them directly — O(k) per step. The heap turns that into O(log k) per step, by maintaining the "which is smallest" answer incrementally rather than recomputing it from scratch every time.
- **Complexity:** O(N log k), where N is the TOTAL number of elements across all lists and k is the number of lists. Each of the N elements is pushed and popped from a heap of size at most k. Compare to concatenating everything and sorting from scratch at O(N log N) — when k is much smaller than N (a few long lists, rather than many short ones), O(N log k) is a meaningful improvement.

### Common Pitfalls
- Forgetting to track which LIST an element came from (only storing the value) — without that bookkeeping, there's no way to know which list to pull the next element from after popping the current minimum.
- Seeding the heap with entire lists instead of just their first elements — this defeats the purpose; the heap should only ever hold up to k elements at a time (one "current front" per list), not all N elements at once.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>
using namespace std;

// K-WAY MERGE: merge k already-sorted lists into one sorted result, using a min-heap
// to always know which list's NEXT element is currently smallest across all k lists --
// without that heap, you'd need to compare all k "current front" elements directly
// every step, which is O(k) per step instead of O(log k).
vector<int> mergeKLists(vector<vector<int>>& lists) {
    // heap holds {value, listIndex, elementIndex} -- enough info to find the NEXT
    // element in the same list once this one is consumed
    priority_queue<tuple<int,int,int>, vector<tuple<int,int,int>>, greater<>> minHeap;

    for (int i = 0; i < (int)lists.size(); i++) {
        if (!lists[i].empty()) {
            minHeap.push({lists[i][0], i, 0});   // seed the heap with each list's first element
        }
    }

    vector<int> result;
    while (!minHeap.empty()) {
        auto [val, listIdx, elemIdx] = minHeap.top();
        minHeap.pop();
        result.push_back(val);

        if (elemIdx + 1 < (int)lists[listIdx].size()) {
            // this list has more elements -- push its NEXT element, from the SAME list
            minHeap.push({lists[listIdx][elemIdx + 1], listIdx, elemIdx + 1});
        }
    }
    return result;
}

int main() {
    vector<vector<int>> lists{
        {1, 4, 7},
        {2, 5, 8, 11},
        {3, 6}
    };

    auto merged = mergeKLists(lists);
    cout << "merged: ";
    for (int x : merged) cout << x << " ";
    cout << "\n";

    // COMPLEXITY: O(N log k), where N is the TOTAL number of elements across all lists,
    // and k is the number of lists. Each of the N elements gets pushed and popped from
    // a heap of size at most k, so O(log k) work per element. Compare to concatenating
    // everything and sorting from scratch: O(N log N) -- when k is much smaller than N
    // (few lists, but each one long), O(N log k) is meaningfully better than O(N log N).
    // This is the natural extension of 02_Sliding_Window_and_Prefix_Sum's two-pointer
    // merge (which only handled 2 lists at a time) to an arbitrary number of lists at once.

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. When a Heap Is (and Isn't) the Right Tool

### The Why
Heaps are excellent at one specific thing — repeatedly accessing and removing the current extreme (min or max) element as data changes — and genuinely bad at things that sound superficially similar but aren't the same operation.

### Core Mechanism
- **A heap is the right tool when:** you repeatedly need "the current min/max" from a changing collection, especially interleaved with insertions (top-k, running median, k-way merge, and — a preview of the graphs phase — Dijkstra's algorithm, which repeatedly needs "the closest unvisited node so far").
- **A heap is the WRONG tool when:** you need to search for an ARBITRARY element (not just the current extreme) — a heap gives you no better than O(n) for "does this specific value exist," since heap order only guarantees parent/child relationships, not any useful ordering between siblings or across the whole structure. A hash set (`03_Hashing_and_Binary_Search`) or a sorted structure (`set`/`map`) is the right tool for that instead.
- **A heap is also the wrong tool when:** you need the FULL sorted order of everything, not just repeated access to one extreme at a time — at that point, you're better off just sorting directly (`04_Sorting_Algorithms`) rather than popping a heap n times (which is literally heap sort, and — as covered in that notebook — usually slower in practice than a well-implemented quicksort due to cache locality).
- **A fast self-check:** if the problem's core operation is "give me the current best/worst, repeatedly, as things change," lean heap. If it's "does X exist" or "give me everything in order," lean hash-based structures or direct sorting instead.

### Common Pitfalls
- Reaching for a heap to check membership ("has this value been added before?") — this is an O(n) heap scan when a hash set would answer the same question in O(1) average.


## 8. Phase Checkpoint, Cheat Sheet, and Answer Key

### Heap Pattern Cheat Sheet

| Pattern | Structure | Complexity | Example |
|---|---|---|---|
| Build-heap | Bottom-up heapify | O(n) build | Heap sort's build step, or any "heapify an existing array" |
| Custom comparator | `operator<` or lambda + `decltype` | O(log n) per push/pop | Task scheduling, any struct with a priority field |
| Top-k | Size-bounded min-heap (for "largest") | O(n log k) | Kth Largest Element, Top K Frequent Elements |
| Two-heap median | Max-heap (lower) + min-heap (upper), size-balanced | O(log n) per insertion, O(1) per query | Find Median from Data Stream |
| K-way merge | Min-heap seeded with k "current front" elements | O(N log k) | Merge k Sorted Lists |

### Checkpoint Questions
1. Why is build-heap O(n) rather than O(n log n), given each individual heapify call can cost up to O(log n)?
2. For a `priority_queue<Task>` where lower `priority` numbers should come out first, why does `operator<` need to compare `priority > other.priority` instead of the seemingly more natural `priority < other.priority`?
3. In the top-k pattern for finding the Kth LARGEST element, why does the heap need to be a MIN-heap rather than a max-heap?
4. What invariant does the two-heap median technique maintain after every insertion, and what breaks if it's violated?
5. In k-way merge, what specific piece of information must be stored alongside each heap entry, beyond just the value itself, and why?
6. Give one example of a problem where a heap is clearly the WRONG tool, and name the better alternative.

### Answer Key
1. Most nodes in a complete binary tree sit near the bottom (leaves and near-leaves), where a heapify call can sink at most a small number of levels — only a small fraction of nodes are near the root, where the expensive O(log n) sink could even happen. Summing "nodes at height h" times "h" across all heights converges to O(n) rather than O(n log n), because the cheap, low-height calls vastly outnumber the expensive, high-height ones.
2. `priority_queue` is a MAX-heap by default — `.top()` returns whatever `operator<` considers "biggest." To make the smallest priority NUMBER come out first (most urgent), that smallest number needs to be treated as the "biggest" by the comparison — which means a Task is considered "less than" another when its priority number is actually bigger, i.e. `priority > other.priority`.
3. The heap holds the current top-k candidates, and the operation needed repeatedly is "find and evict the WEAKEST of those candidates" when a stronger one arrives — that's the minimum of the top-k set, which a min-heap exposes in O(1) via `.top()`. A max-heap would give fast access to the strongest candidate instead, which isn't what's needed to decide evictions.
4. `lowerHalf` (max-heap) must have a size either equal to or exactly one greater than `upperHalf` (min-heap) — never behind, never more than one ahead. If violated, the O(1) median formula breaks, since it depends on knowing exactly which heap (if either) holds the single "extra" middle element in an odd-sized dataset.
5. Which list the element came from, and its position within that list — without this, after popping an element from the heap, there's no way to know which list's next element should be pushed in to replace it.
6. Checking whether a specific value exists in a collection ("has X been seen before") — a heap only guarantees parent/child ordering, not any useful relationship between arbitrary pairs of elements, so membership checking is O(n) in a heap. A hash set (`03_Hashing_and_Binary_Search`) answers the same question in O(1) average.

### Mini Project
Implement a "task scheduler" simulation: given a list of tasks, each with a priority and a processing time, simulate processing them in priority order (using Section 3's custom comparator), and track the CURRENT median processing time of all tasks processed so far after each one completes (using Section 5's two-heap technique).

This exercises: custom comparators for a real-world-shaped struct, AND the two-heap running-median technique, combined in one simulation — a realistic example of multiple heap-based patterns from this notebook working together rather than in isolation.


## 9. LeetCode Practice Problems

Grouped by pattern. Hints point at the pattern's key decision, not the full solution.

### Basic Heap Usage

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 703 | Kth Largest Element in a Stream | Easy | Section 4's top-k pattern, but maintained incrementally as a class instead of computed once |
| 1046 | Last Stone Weight | Easy | Repeatedly pop the two largest, push back their difference — a direct max-heap simulation |

### Custom Comparators

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 1642 | Furthest Building You Can Reach | Medium | Greedily use ladders on the biggest jumps, bricks on the rest — a min-heap of "ladder-assigned jumps so far" lets you swap out the smallest ladder-assigned jump for bricks when you run out of ladders |
| 767 | Reorganize String | Medium | Max-heap by character frequency; always place the currently-most-frequent character that ISN'T the same as the previous placement |

### Top-K Pattern

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 215 | Kth Largest Element in an Array | Medium | Section 4, directly |
| 347 | Top K Frequent Elements | Medium | Count frequencies with a hash map (`03_Hashing_and_Binary_Search`), then apply the top-k heap pattern on top of the counts |
| 973 | K Closest Points to Origin | Medium | Same top-k shape, ordering by squared distance instead of raw value |

### Two-Heap Technique

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 295 | Find Median from Data Stream | Hard | Section 5, directly |
| 480 | Sliding Window Median | Hard | Combines Section 5's two-heap technique with `02_Sliding_Window_and_Prefix_Sum`'s window — the added difficulty is REMOVING an element that's leaving the window, which a plain heap can't do efficiently; research "lazy deletion" as the standard workaround |

### K-Way Merge

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 23 | Merge k Sorted Lists | Hard | Section 6, directly (applied to linked lists — you'll formalize those in `07_Linked_Lists`, but the heap logic is identical) |
| 378 | Kth Smallest Element in a Sorted Matrix | Medium | Treat each matrix row as one of the k sorted lists in a k-way merge, stopping after extracting k elements instead of merging everything |

### Self-Check Before Moving to `02_Trees_and_BST`
If you can explain, without looking back at this notebook, why build-heap is O(n) and why top-k uses a min-heap for the LARGEST elements, you've internalized the two ideas in this notebook most likely to be tested directly. `02_Trees_and_BST` moves from heaps (a narrowly-ordered, pointer-free tree) to general binary trees and BSTs, where the ordering rules are richer and pointer-based structure returns — the README's Sections 1–2 are your existing reference for that material; this course phase will build hands-on practice on top of it.
